# **Project Name**    - Zomato Restaurant Clustering and Sentiment Analysis

##### **Project Type**    - Unsupervised (restaurant clustering) + NLP sentiment analysis
##### **Contribution**    - Individual
##### **Team Member 1 -** ✏️ (write your name here)

# **Project Summary -**

Zomato is an Indian restaurant-discovery and food-delivery platform that lists restaurants, menus and customer reviews. This project analyses Zomato data for 100 restaurants in Hyderabad to help customers find the right restaurant and to help the company see which kinds of restaurants it has and where it is lagging.

**Data.** Two files were used: a restaurant table (105 restaurants with name, cost for one person, cuisines, collection tags and timings) and a review table (10,000 reviews, exactly 100 for each of 100 restaurants, with reviewer, text, rating, reviewer followers, time and number of pictures). The reviews cover May 2016 to May 2019, with most of them from 2018 and 2019.

**Cleaning and wrangling.** Cost was stored as text ("1,300"), ratings contained a stray value ("Like"), the reviewer metadata packed two numbers into one string, and 36 duplicate rows and 38 almost-empty rows existed. All were fixed. Five restaurants without any reviews were left out of the review-based analysis. Sentiment of every review was scored with VADER and the reviews were summarised into one row per restaurant (average rating, average sentiment, share of negative reviews, pictures per review, review length, reviewer followers).

**EDA and storytelling.** Fifteen charts were built following the Univariate, Bivariate, Multivariate (UBM) rule. Ratings are strongly skewed to the top: about 38% of reviews are 5 stars and only about a quarter are 2 stars or below. The median cost for one person is Rs. 700 and North Indian and Chinese are by far the most common cuisines. Reviews are posted mostly late in the evening (9-11 pm) and at weekends. Premium restaurants (cost above Rs. 1,000) have the highest average rating (3.95 versus 3.36 for budget places).

**Hypothesis tests.** Three statistical tests were run. Cost and average rating are positively related (Spearman rho = 0.38, p < 0.001). Reviews with pictures give higher ratings (3.86 versus 3.54, p = 0.003). Experienced "critic" reviewers do not rate differently from other reviewers (p = 0.52).

**Feature engineering and pre-processing.** Outliers were capped with the IQR rule, skewed features were log-transformed, and all features were standardised. Cuisines were multi-hot encoded. For the review text, the full NLP chain was applied: contraction expansion, lower-casing, punctuation/URL/number removal, stop-word removal (keeping negations), slang clean-up, tokenisation, lemmatisation and TF-IDF vectorisation.

**Models.** Three clustering models were built and tuned on the restaurant features: K-Means, Agglomerative (hierarchical) clustering and a Gaussian Mixture Model. They were compared with the silhouette score, Davies-Bouldin index, Calinski-Harabasz index and a resampling stability test. The final choice was K-Means with ✏️ [K] clusters (silhouette ≈ ✏️ [value]), because it is simple to explain, stable and can score new restaurants. Each cluster was profiled and turned into a business strategy for customers and for Zomato. As a bonus, three supervised text classifiers (Logistic Regression, Naive Bayes, Linear SVM) were cross-validated and tuned to predict whether a review is positive or negative; the best reached a macro-F1 of about 0.95 ✏️ [check your output]. Feature importance showed which words and which restaurant features drive the results.

**Outcome.** The segments help customers pick restaurants by price and quality, and help Zomato decide where to promote, where to support quality, and where to feature hidden gems. Both models were saved with joblib and tested on unseen data.

# **GitHub Link -**

✏️ Paste your GitHub link here, for example: https://github.com/your-username/zomato-restaurant-clustering

# **Problem Statement**


**Zomato lists thousands of restaurants and reviews, but customers struggle to find the best restaurant for their budget and taste, and the company needs to know which segments of restaurants it has and where it is lagging.**

Using two datasets (restaurant metadata and customer reviews), this project:
1. cleans and explores the data and tells the story with at least 15 charts,
2. analyses the **sentiment** of customer reviews,
3. **clusters** the restaurants into meaningful segments using unsupervised machine learning,
4. turns each segment into a practical strategy for customers and for Zomato, and
5. builds a model that predicts whether a new review is positive or negative.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Install the sentiment analyser (VADER). Safe to re-run; "!" runs a shell command in Colab.
!pip install -q vaderSentiment

# ---- Standard libraries ----
import os, re, glob, warnings
import numpy as np
import pandas as pd
import joblib

# ---- Visualization ----
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Statistics ----
from scipy.stats import spearmanr, mannwhitneyu
from scipy.cluster.hierarchy import linkage, dendrogram

# ---- Machine learning ----
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, ConfusionMatrixDisplay)

# ---- Text: VADER sentiment + (optional) NLTK for lemmatization / POS tagging ----
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
try:
    import nltk
    for pkg in ["wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
        nltk.download(pkg, quiet=True)
    from nltk.stem import WordNetLemmatizer
except Exception as err:          # the notebook still runs without NLTK (simple fallbacks are used)
    print("NLTK not available, using fallbacks:", err)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)
RANDOM_STATE = 42     # fixed seed -> reproducible results

### Dataset Loading

In [ ]:
# Load Dataset
# The files are found by keyword, so it works whatever the exact file name is
# (Colab may rename uploads, e.g. spaces instead of underscores).
def find_csv(keywords):
    for f in glob.glob("*.csv"):
        if any(k in f.lower() for k in keywords):
            return f
    return None

if find_csv(["review"]) is None or find_csv(["meta", "names"]) is None:
    try:
        from google.colab import files
        print("Upload BOTH csv files (reviews + names/metadata):")
        files.upload()
    except ImportError:
        raise FileNotFoundError("Put both CSV files in the same folder as this notebook.")

REVIEWS_PATH = find_csv(["review"])
META_PATH = find_csv(["meta", "names"])
print("Using:", REVIEWS_PATH, "|", META_PATH)

reviews_raw = pd.read_csv(REVIEWS_PATH)
meta_raw = pd.read_csv(META_PATH)

### Dataset First View

In [ ]:
# Dataset First Look
print("Restaurant names & metadata:")
display(meta_raw.head())
print("Reviews:")
display(reviews_raw.head())

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print("Restaurant metadata -> rows:", meta_raw.shape[0], "| columns:", meta_raw.shape[1])
print("Reviews             -> rows:", reviews_raw.shape[0], "| columns:", reviews_raw.shape[1])

In [ ]:
# How do the two tables connect? Reviews[Restaurant] -> Metadata[Name]
names_meta = set(meta_raw["Name"].str.strip())
names_rev = set(reviews_raw["Restaurant"].str.strip())
print("Restaurants in the review table      :", len(names_rev))
print("Restaurants in the metadata table    :", len(names_meta))
print("Reviewed restaurants found in metadata:", len(names_rev & names_meta))
print("Restaurants with no reviews          :", sorted(names_meta - names_rev))
print("Reviews per restaurant (min / max)   :", reviews_raw.groupby("Restaurant").size().min(), "/", reviews_raw.groupby("Restaurant").size().max())

### Dataset Information

In [ ]:
# Dataset Info
print("=== Restaurant metadata ===")
meta_raw.info()
print("\n=== Reviews ===")
reviews_raw.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print("Duplicate rows in restaurant metadata:", meta_raw.duplicated().sum())
print("Duplicate rows in reviews            :", reviews_raw.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
for name, df in [("Restaurant metadata", meta_raw), ("Reviews", reviews_raw)]:
    miss = df.isna().sum()
    print(f"=== {name} ===")
    display(pd.DataFrame({"missing": miss, "percent": (miss / len(df) * 100).round(2)}))

In [ ]:
# Visualizing the missing values
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.heatmap(meta_raw.isna(), cbar=False, yticklabels=False, ax=axes[0])
axes[0].set_title("Missing values - restaurant metadata (white = present, dark = missing)")
sns.heatmap(reviews_raw.isna(), cbar=False, yticklabels=False, ax=axes[1])
axes[1].set_title("Missing values - reviews")
plt.tight_layout()
plt.show()

### What did you know about your dataset?

- **Two tables.** The restaurant table has 105 restaurants (6 columns). The review table has 10,000 reviews (7 columns) for 100 restaurants, exactly 100 reviews each. Five restaurants have no reviews.
- **Duplicates:** none in the restaurant table; **36 duplicate rows** in the reviews.
- **Missing values:** *Collections* is missing for about half of the restaurants (54 of 105), which just means "no collection tag". *Timings* is missing once. In the reviews, 38 rows are almost completely empty (no reviewer, rating, metadata, time) and 45 rows have no review text.
- **Data type problems:** *Cost* is text with commas, *Rating* is text and contains one wrong value ("Like"), *Metadata* packs "number of reviews" and "number of followers" into one string, and *Time* is text.
- **Ratings** run from 1 to 5, mostly whole stars with a few half-stars.
- The data is from Hyderabad and the reviews are from May 2016 to May 2019.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print("Restaurant metadata columns:", list(meta_raw.columns))
print("Review columns             :", list(reviews_raw.columns))

In [ ]:
# Dataset Describe
print("=== Restaurant metadata ===")
display(meta_raw.describe(include="all").T)
print("=== Reviews ===")
display(reviews_raw.describe(include="all").T)

### Variables Description

**Restaurant table**
| Column | Meaning |
|---|---|
| Name | Restaurant name (unique key) |
| Links | Zomato page of the restaurant |
| Cost | Estimated cost of dining for one person (Rs.) |
| Collections | Zomato tags such as "Top-Rated", "Great Buffets" (comma separated, often empty) |
| Cuisines | Cuisines served (comma separated) |
| Timings | Opening hours (free text) |

**Review table**
| Column | Meaning |
|---|---|
| Restaurant | Which restaurant the review is about (links to *Name*) |
| Reviewer | Name of the reviewer |
| Review | Review text |
| Rating | Stars given (1 to 5) |
| Metadata | Reviewer's number of reviews and followers, e.g. "96 Reviews , 1069 Followers" |
| Time | Date and time of the review |
| Pictures | Number of pictures posted with the review |

**Created by me:** reviewer_reviews, reviewer_followers, review_len, has_picture, is_critic, sentiment (VADER), and per-restaurant averages (avg_rating, avg_sentiment, pct_negative, avg_pictures, avg_review_words, avg_reviewer_followers).

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for name, df in [("Restaurant metadata", meta_raw), ("Reviews", reviews_raw)]:
    print(f"=== {name} ===")
    print(df.nunique().to_string())
    print()

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

# ------------------------------------------------------------------
# 1) Helper functions that clean each table
# ------------------------------------------------------------------
def clean_reviews(df):
    """Return a cleaned copy of the reviews table."""
    df = df.copy()
    df["Restaurant"] = df["Restaurant"].str.strip()

    # remove exact duplicate rows
    df = df.drop_duplicates()

    # Rating is text -> numeric. The stray value "Like" becomes NaN; rows with no rating are unusable.
    df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
    df = df.dropna(subset=["Rating"])

    # Metadata looks like "96 Reviews , 1069 Followers" -> two numeric columns
    meta_txt = df["Metadata"].fillna("").str.replace(",", "", regex=False)
    df["reviewer_reviews"] = pd.to_numeric(meta_txt.str.extract(r"(\d+)\s*Review")[0], errors="coerce").fillna(0)
    df["reviewer_followers"] = pd.to_numeric(meta_txt.str.extract(r"(\d+)\s*Follower")[0], errors="coerce").fillna(0)

    # date/time features
    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    df["year"] = df["Time"].dt.year

    # review text: missing -> empty string, length measured in words
    df["Review"] = df["Review"].fillna("").astype(str).str.strip()
    df["review_len"] = df["Review"].str.split().str.len()
    return df.drop(columns=["Metadata"]).reset_index(drop=True)


def clean_metadata(df):
    """Return a cleaned copy of the restaurant table."""
    df = df.copy()
    df["Name"] = df["Name"].str.strip()
    df = df.drop_duplicates(subset="Name")

    # Cost is text with commas ("1,300") -> number
    df["Cost"] = pd.to_numeric(df["Cost"].astype(str).str.replace(",", "", regex=False), errors="coerce")

    # A missing collection tag is a real state ("no tag"), not an error
    df["Collections"] = df["Collections"].fillna("")
    df["Cuisines"] = df["Cuisines"].fillna("")

    # comma-separated text -> lists and counts
    df["cuisine_list"] = df["Cuisines"].apply(lambda s: [c.strip() for c in s.split(",") if c.strip()])
    df["n_cuisines"] = df["cuisine_list"].str.len()
    df["n_collections"] = df["Collections"].apply(lambda s: len([c for c in s.split(",") if c.strip()]))
    return df.reset_index(drop=True)


reviews = clean_reviews(reviews_raw)
meta = clean_metadata(meta_raw)

# ------------------------------------------------------------------
# 2) Derived columns used later in the analysis
# ------------------------------------------------------------------
reviews["has_picture"] = reviews["Pictures"] > 0
reviews["weekday"] = reviews["Time"].dt.day_name()
reviews["hour"] = reviews["Time"].dt.hour
# "Critic" = experienced, well-followed reviewer (10+ reviews and 500+ followers)
reviews["is_critic"] = (reviews["reviewer_followers"] >= 500) & (reviews["reviewer_reviews"] >= 10)

# Only rows that contain review text are used for sentiment / NLP
reviews_text = reviews[reviews["review_len"] > 0].copy()
reviews_text["rating_group"] = pd.cut(reviews_text["Rating"], bins=[0, 2.5, 3.5, 5],
                                      labels=["Low (1-2.5)", "Mid (3-3.5)", "High (4-5)"])

# ------------------------------------------------------------------
# 3) Sentiment with VADER (on the RAW text: VADER uses capitals, "!!!" and emojis as signals)
#    compound score: -1 (very negative) ... +1 (very positive)
# ------------------------------------------------------------------
sia = SentimentIntensityAnalyzer()
reviews_text["sentiment"] = reviews_text["Review"].apply(lambda t: sia.polarity_scores(t)["compound"])
reviews_text["sentiment_label"] = np.where(reviews_text["sentiment"] >= 0.05, "Positive",
                                  np.where(reviews_text["sentiment"] <= -0.05, "Negative", "Neutral"))

# ------------------------------------------------------------------
# 4) One row per restaurant = metadata + summary of its reviews
# ------------------------------------------------------------------
agg_all = reviews.groupby("Restaurant").agg(avg_rating=("Rating", "mean"),
                                            avg_pictures=("Pictures", "mean"),
                                            avg_reviewer_followers=("reviewer_followers", "mean"),
                                            n_reviews=("Rating", "size"))
agg_text = reviews_text.groupby("Restaurant").agg(avg_sentiment=("sentiment", "mean"),
                                                  pct_negative=("sentiment_label", lambda s: (s == "Negative").mean() * 100),
                                                  avg_review_words=("review_len", "mean"))
restaurants = (meta.merge(agg_all, left_on="Name", right_index=True, how="inner")
                   .merge(agg_text, left_on="Name", right_index=True, how="left"))
restaurants["cost_bucket"] = pd.cut(restaurants["Cost"], bins=[0, 500, 1000, np.inf],
                                    labels=["Budget (<=500)", "Mid (501-1000)", "Premium (>1000)"])

# one row per (restaurant, cuisine) - handy for cuisine charts
cuisine_long = (restaurants[["Name", "Cost", "avg_rating", "cuisine_list"]]
                .explode("cuisine_list").rename(columns={"cuisine_list": "Cuisine"}))
cuisine_counts = cuisine_long["Cuisine"].value_counts()

print("Reviews (clean)        :", reviews.shape)
print("Reviews with text      :", reviews_text.shape)
print("Restaurants (analysis) :", restaurants.shape)
print("Restaurants without reviews (dropped from the merge):",
      sorted(set(meta["Name"]) - set(reviews["Restaurant"])))
display(restaurants.drop(columns=["Links", "Collections", "Timings", "cuisine_list"]).head())

### What all manipulations have you done and insights you found?

**Manipulations**
1. Removed 36 duplicate reviews and the rows without a rating (38 almost-empty rows plus the "Like" row) - they cannot be used for rating or sentiment analysis. 9,961 reviews remain, 9,954 of them with text.
2. Converted *Cost* ("1,300") and *Rating* to numbers and *Time* to a date-time.
3. Split *Metadata* into **reviewer_reviews** and **reviewer_followers** (missing followers = 0).
4. Counted cuisines and collections per restaurant (**n_cuisines**, **n_collections**) and turned the comma-separated text into lists.
5. Added **has_picture**, weekday, hour and an **is_critic** flag (10+ reviews and 500+ followers).
6. Scored every review with **VADER** to get a sentiment score and label (Positive / Neutral / Negative).
7. Built one row per restaurant by combining the metadata with review statistics (100 restaurants remain after the merge).

**Insights found**
- Every restaurant has exactly 100 reviews, so the data looks like the latest 100 reviews of each restaurant. That is why almost all reviews are from 2018-2019.
- Only 20% of reviews contain pictures and about 8% are written by "critics".
- About a quarter of reviewers have written one review or none, so most reviewers are casual customers.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1 - Distribution of review ratings (Univariate)

In [ ]:
# Chart - 1 visualization code : distribution of star ratings (univariate)
order = sorted(reviews["Rating"].unique())
ax = sns.countplot(data=reviews, x="Rating", order=order, color="#4C72B0")
ax.set_title("Distribution of review ratings")
ax.set_xlabel("Rating (stars)"); ax.set_ylabel("Number of reviews")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.show()
print((reviews["Rating"].value_counts(normalize=True).sort_index() * 100).round(1).to_string())

##### 1. Why did you pick the specific chart?

A count plot (bar chart) is the natural way to show how many reviews fall into each discrete star value, and it shows the shape of the rating distribution at a glance.

##### 2. What is/are the insight(s) found from the chart?

Ratings are strongly skewed to the top: about 38% of reviews are 5 stars and about 24% are 4 stars, so over 60% of reviews are 4-5 stars. At the other end, about 17% are 1 star, which is far more than 2 stars (about 7%), so people who are unhappy tend to be *very* unhappy. Half-star ratings are rare (about 1.5%).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive:** most customers are satisfied, which supports Zomato's brand and the restaurants' visibility. **Negative:** the large block of 1-star reviews is a real risk: a visible group of very unhappy customers can hurt trust in a restaurant and in the platform. Because the data is skewed, accuracy alone is a poor measure for the sentiment model (see the class-imbalance section).

#### Chart - 2 - Cost for one person (Univariate)

In [ ]:
# Chart - 2 visualization code : distribution of cost for one person (univariate)
med = restaurants["Cost"].median()
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(restaurants["Cost"], bins=20, kde=True, color="#55A868", ax=ax)
ax.axvline(med, color="red", ls="--", label=f"Median = Rs.{med:.0f}")
ax.set_title("Cost for one person across restaurants")
ax.set_xlabel("Cost (Rs.)"); ax.set_ylabel("Number of restaurants"); ax.legend()
plt.show()
print(restaurants["Cost"].describe().round(0).to_string())

##### 1. Why did you pick the specific chart?

A histogram with a density curve shows how a numeric variable is spread, and the median line shows the typical price. It is the standard univariate chart for a numeric column such as cost.

##### 2. What is/are the insight(s) found from the chart?

The median cost for one person is Rs. 700 (mean about Rs. 870). Half of the restaurants cost between Rs. 500 and Rs. 1,200 and the range is Rs. 150 to Rs. 2,800. The distribution is right-skewed: a few very expensive restaurants stretch the tail.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive:** Zomato can design price filters and promotions around the dense Rs. 500-1,200 range where most restaurants and customers are. **Negative:** the few very expensive restaurants have a small audience, and averages are pulled up by them, so decisions should use the median and not the mean.

#### Chart - 3 - Most common cuisines (Univariate)

In [ ]:
# Chart - 3 visualization code : most common cuisines (univariate)
top12 = cuisine_counts.head(12)
plt.figure(figsize=(9, 5))
sns.barplot(x=top12.values, y=top12.index, color="#4C72B0")
plt.title("Top 12 cuisines (number of restaurants serving them)")
plt.xlabel("Number of restaurants"); plt.ylabel("")
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar chart is best for ranking categories with long names (cuisine names). Sorting the bars makes the ranking obvious.

##### 2. What is/are the insight(s) found from the chart?

North Indian (about 60 of 100 restaurants) and Chinese (about 40) dominate, followed by Continental, Biryani, Asian, Italian, Desserts and Fast Food. Specialised cuisines such as Kebab, American or Mughlai appear in only a handful of restaurants.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive:** customers will always find North Indian and Chinese options, and Zomato can use them as the default categories. **Negative / opportunity:** niche cuisines are under-represented, so customers looking for them have little choice. Zomato could onboard more restaurants in those niches to attract new customers.

#### Chart - 4 - Reviews per month (Univariate, time)

In [ ]:
# Chart - 4 visualization code : number of reviews per month (univariate, time)
monthly = reviews["Time"].dt.to_period("M").value_counts().sort_index()
monthly.index = monthly.index.to_timestamp()
ax = monthly.plot(figsize=(11, 4), color="#C44E52")
ax.set_title("Number of reviews per month"); ax.set_xlabel(""); ax.set_ylabel("Reviews")
plt.show()
print("Busiest month:", monthly.idxmax().strftime("%b %Y"), "with", monthly.max(), "reviews")

##### 1. Why did you pick the specific chart?

A line chart is the correct choice for showing a trend over time. Grouping the reviews by month makes the time pattern easy to see.

##### 2. What is/are the insight(s) found from the chart?

Review volume is tiny in 2016-2017 (about 250 reviews in total) and jumps in 2018-2019 (about 4,900 and 4,800 reviews). The busiest month is May 2019 with 1,346 reviews. Since each restaurant has exactly 100 reviews, the early months are thin simply because the data keeps only each restaurant's latest 100 reviews.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive:** recent reviews dominate, so the insights describe the *current* customer experience. **Negative:** the data cannot be used to measure long-term growth or seasonality, because the early period is hardly covered. Any trend conclusion from this chart would be misleading, which is an important limitation to state.

#### Chart - 5 - Sentiment of the reviews (Univariate)

In [ ]:
# Chart - 5 visualization code : sentiment of the reviews (univariate)
order = ["Positive", "Neutral", "Negative"]
ax = sns.countplot(data=reviews_text, x="sentiment_label", order=order, color="#8172B2")
ax.set_title("VADER sentiment label of reviews"); ax.set_xlabel(""); ax.set_ylabel("Number of reviews")
total = len(reviews_text)
for p in ax.patches:
    ax.annotate(f"{p.get_height() / total * 100:.1f}%", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.show()

##### 1. Why did you pick the specific chart?

A count plot with percentage labels shows the share of positive, neutral and negative reviews, i.e. the overall customer mood, in one picture.

##### 2. What is/are the insight(s) found from the chart?

✏️ (check your chart for the exact percentages) The majority of reviews are positive, a smaller share is neutral and only a minority is negative. This matches the star ratings, where most reviews are 4-5 stars.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive:** a mostly positive sentiment is good for the platform. **Negative:** the negative reviews, even if fewer, contain the specific complaints (service, cold food, waiting time). Zomato and restaurants can mine them to fix problems that cost customers.

#### Chart - 6 - Average rating by cuisine (Bivariate)

In [ ]:
# Chart - 6 visualization code : average rating by cuisine (bivariate: categorical - numerical)
top10 = cuisine_counts.head(10).index
cr = (cuisine_long[cuisine_long["Cuisine"].isin(top10)]
      .groupby("Cuisine")["avg_rating"].mean().sort_values())
plt.figure(figsize=(9, 5))
ax = sns.barplot(x=cr.values, y=cr.index, color="#55A868")
ax.set_xlim(0, 5); ax.set_xlabel("Average of restaurant ratings"); ax.set_ylabel("")
ax.set_title("Average rating of restaurants by cuisine (10 most common cuisines)")
for i, v in enumerate(cr.values):
    ax.text(v + 0.03, i, f"{v:.2f}", va="center")
plt.show()

##### 1. Why did you pick the specific chart?

A bar chart of the mean rating per category compares cuisines on the same scale. The axis starts at 0 so that the bars are not exaggerated.

##### 2. What is/are the insight(s) found from the chart?

Average ratings are close to each other, between about 3.3 and 3.9. Asian (about 3.9), Continental (3.8) and Kebab (3.8) have the highest averages, while Fast Food (3.3), Biryani (3.4) and Chinese (3.5) have the lowest.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive:** Zomato can highlight the best-rated cuisines in recommendations. **Negative:** the cuisines with the lowest averages are also among the most popular ones (Chinese, Biryani, Fast Food), so many customers are exposed to weaker experiences. Quality programmes for those cuisines would give the biggest gain. The differences are small, so cuisine alone does not decide quality.

#### Chart - 7 - Cost vs average rating (Bivariate)

In [ ]:
# Chart - 7 visualization code : cost vs average rating (bivariate: numerical - numerical)
rho, p = spearmanr(restaurants["Cost"], restaurants["avg_rating"])
plt.figure(figsize=(9, 5))
sns.regplot(data=restaurants, x="Cost", y="avg_rating", scatter_kws={"alpha": 0.7}, line_kws={"color": "red"})
plt.title(f"Cost vs average rating (Spearman rho = {rho:.2f}, p = {p:.4f})")
plt.xlabel("Cost for one person (Rs.)"); plt.ylabel("Average rating")
plt.show()

##### 1. Why did you pick the specific chart?

A scatter plot with a regression line is the standard way to see the relationship between two numeric variables, and each dot is one restaurant. Spearman correlation is used because it does not assume a straight line or normal data.

##### 2. What is/are the insight(s) found from the chart?

There is a moderate positive relationship: more expensive restaurants tend to have higher average ratings (Spearman rho about 0.38, p < 0.001). The spread is wide, so there are cheap restaurants with excellent ratings and expensive ones with poor ratings.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive:** it supports a 'premium quality' story and helps Zomato recommend the right price tier. The cheap-but-excellent restaurants are the 'hidden gems' that can be promoted to value-conscious customers. **Negative:** expensive restaurants that are rated poorly are high-risk for customer disappointment and for refunds or complaints.

#### Chart - 8 - Review length by rating (Bivariate)

In [ ]:
# Chart - 8 visualization code : review length by rating (bivariate: numerical - categorical)
plt.figure(figsize=(10, 5))
sns.boxplot(data=reviews_text, x="Rating", y="review_len", showfliers=False, color="#CCB974")
plt.title("Review length (words) for each star rating")
plt.xlabel("Rating (stars)"); plt.ylabel("Words in review")
plt.show()
print(reviews_text.groupby("rating_group")["review_len"].median().to_string())

##### 1. Why did you pick the specific chart?

A box plot shows the median and spread of a numeric variable (words in a review) for each category (star rating), and hiding outliers keeps the chart readable.

##### 2. What is/are the insight(s) found from the chart?

Middle ratings write the longest reviews: the median review has about 41 words for 3-3.5 stars, versus about 34 words for low (1-2.5) and 33 words for high (4-5) ratings. People who are very happy or very unhappy write short, emotional reviews, while people with mixed feelings explain themselves.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive:** the long mid-rating reviews are the most informative about what to improve, so they are worth mining first. **Negative:** the short 1-star reviews carry little detail ('worst', 'bad'), so restaurants learn little from them. Zomato could prompt unhappy customers with questions to collect actionable feedback.

#### Chart - 9 - Rating group vs sentiment label (Bivariate)

In [ ]:
# Chart - 9 visualization code : rating group vs sentiment label (bivariate: categorical - categorical)
ct = pd.crosstab(reviews_text["rating_group"], reviews_text["sentiment_label"], normalize="index") * 100
ct = ct[["Positive", "Neutral", "Negative"]]
plt.figure(figsize=(7, 4))
sns.heatmap(ct, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% of reviews in the rating group"})
plt.title("Do stars and text sentiment agree?"); plt.xlabel("VADER sentiment"); plt.ylabel("Star rating group")
plt.show()

##### 1. Why did you pick the specific chart?

A heatmap of a normalised cross-table is the clearest way to compare two categorical variables. Each row shows how the reviews of one rating group are split over the sentiment labels.

##### 2. What is/are the insight(s) found from the chart?

✏️ (read the percentages in your heatmap) High-rated reviews should be overwhelmingly positive in text, while low-rated reviews are a mix of negative, neutral and even positive text. The mismatch shows that star ratings and text do not always agree.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive:** where text and stars agree, sentiment analysis is a good, cheap quality signal. **Negative:** low-star reviews with positive text may be rating mistakes or sarcasm. If the stars alone are used, a restaurant could be unfairly punished. Combining text sentiment with stars gives a fairer score.

#### Chart - 10 - Ratings with and without pictures (Bivariate)

In [ ]:
# Chart - 10 visualization code : rating with and without pictures (bivariate)
plot_df = reviews.assign(Pictures_in_review=np.where(reviews["has_picture"], "With picture(s)", "No picture"))
plt.figure(figsize=(7, 5))
ax = sns.barplot(data=plot_df, x="Pictures_in_review", y="Rating", errorbar=("ci", 95), color="#4C72B0")
ax.set_ylim(0, 5); ax.set_xlabel(""); ax.set_ylabel("Average rating (with 95% CI)")
ax.set_title("Average rating of reviews with and without pictures")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.show()
print(plot_df["Pictures_in_review"].value_counts().to_string())

##### 1. Why did you pick the specific chart?

A bar chart with 95% confidence intervals compares the mean rating of two groups and shows whether the difference is bigger than the uncertainty.

##### 2. What is/are the insight(s) found from the chart?

Reviews with pictures have a higher average rating (about 3.86) than reviews without (about 3.54). About 20% of reviews (1,983) include a picture. The confidence intervals do not overlap, which fits the significant result in Hypothesis 2.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive:** customers who post pictures are usually happier, and pictures make a listing more attractive. Zomato can encourage pictures in reviews to improve engagement. **Negative:** if pictures are mostly posted by happy customers, the average rating of pictured reviews overstates the real satisfaction, so they should not be the only source for quality scores.

#### Chart - 11 - Best and worst rated restaurants (Bivariate)

In [ ]:
# Chart - 11 visualization code : best and worst rated restaurants (bivariate)
ranked = restaurants.sort_values("avg_rating", ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(x=ranked.head(10)["avg_rating"], y=ranked.head(10)["Name"], color="#55A868", ax=axes[0])
axes[0].set_title("Top 10 restaurants by average rating")
sns.barplot(x=ranked.tail(10)["avg_rating"], y=ranked.tail(10)["Name"], color="#C44E52", ax=axes[1])
axes[1].set_title("Bottom 10 restaurants by average rating")
for ax in axes:
    ax.set_xlim(0, 5); ax.set_xlabel("Average rating"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

Two horizontal bar charts (top 10 and bottom 10) rank restaurants by average rating, which makes the winners and the problem cases easy to read.

##### 2. What is/are the insight(s) found from the chart?

Average ratings range from 2.4 to 4.9. The top restaurants are AB's - Absolute Barbecues (4.88), B-Dubs (4.81) and 3B's - Buddies, Bar & Barbecue (4.76), all premium-priced (Rs. 1,100-1,600). The bottom ones, such as Hotel Zara Hi-Fi (2.40), Asian Meal Box (2.58) and Pakwaan Grand (2.71), are low-cost places (Rs. 200-400).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive:** Zomato can feature the top restaurants and use them as benchmarks. **Negative:** the bottom group are cheap restaurants with persistently low ratings and can damage customers' trust in the platform. They need quality support or a lower priority in search results.

#### Chart - 12 - Critics vs other reviewers (Bivariate)

In [ ]:
# Chart - 12 visualization code : critics vs regular reviewers (bivariate)
plot_df = reviews.assign(Reviewer_type=np.where(reviews["is_critic"], "Critics (10+ reviews, 500+ followers)", "Other reviewers"))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=plot_df, x="Reviewer_type", y="Rating", errorbar=("ci", 95), color="#8172B2", ax=axes[0])
axes[0].set_ylim(0, 5); axes[0].set_title("Average rating given"); axes[0].set_xlabel("")
sns.countplot(data=plot_df, x="Rating", hue="Reviewer_type", ax=axes[1])
axes[1].set_title("How many reviews of each star rating?"); axes[1].legend(title="")
plt.tight_layout(); plt.show()
print(plot_df["Reviewer_type"].value_counts().to_string())

##### 1. Why did you pick the specific chart?

A bar chart of average rating by group (plus a count plot of star ratings) compares two reviewer types. Critics are defined as reviewers with 10+ reviews and 500+ followers.

##### 2. What is/are the insight(s) found from the chart?

Critics are only about 8% of the reviews (759 of 9,961). Their average rating is slightly higher (about 3.80 versus 3.58), but the difference is not statistically significant (p = 0.52, Hypothesis 3). The star distributions are similar.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive:** critics are not harsher than ordinary customers, so a restaurant does not need to fear influencers more than regular guests, and critics' reviews can be used as a trustworthy signal. **Negative:** none found, but because critics have large followings, their rare negative reviews have a bigger reach, so quick responses to them are valuable.

#### Chart - 13 - When are reviews posted? Weekday x hour (Multivariate)

In [ ]:
# Chart - 13 visualization code : when do people post reviews? weekday x hour (multivariate)
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heat = (reviews.pivot_table(index="weekday", columns="hour", values="Rating", aggfunc="size", fill_value=0)
        .reindex(days))
plt.figure(figsize=(13, 4.5))
sns.heatmap(heat, cmap="YlOrRd", cbar_kws={"label": "Number of reviews"})
plt.title("Reviews posted by weekday and hour of the day")
plt.xlabel("Hour of day (24h)"); plt.ylabel("")
plt.show()

##### 1. Why did you pick the specific chart?

A heatmap with weekday on one axis and hour on the other shows how a count changes across two time dimensions at once (a multivariate view), which a simple bar chart cannot do.

##### 2. What is/are the insight(s) found from the chart?

Reviews are posted mostly in the late evening, with a clear peak between 9 pm and 11 pm (the hour with the most reviews is 10 pm), and there is a second smaller bump around 2-3 pm after lunch. Friday to Sunday are the busiest days (Sunday has about 1,800 reviews against about 1,200 on a Tuesday).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive:** Zomato can send review reminders, notifications and promotions at the moments when people are most active (evenings and weekends). **Negative:** very few reviews are written in the early morning and on weekday mornings, so those time slots give little feedback and a weak engagement opportunity.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code (multivariate)
num_cols = ["Cost", "avg_rating", "avg_sentiment", "pct_negative", "avg_pictures",
            "avg_review_words", "avg_reviewer_followers", "n_cuisines", "n_collections"]
plt.figure(figsize=(10, 7))
sns.heatmap(restaurants[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between restaurant-level features")
plt.show()

##### 1. Why did you pick the specific chart?

A correlation heatmap shows the strength and direction of the linear relationship between *all* numeric restaurant features at once, so it is the standard multivariate check. It is also used to spot redundant features before modelling.

##### 2. What is/are the insight(s) found from the chart?

- **avg_rating and avg_sentiment move together** (strong positive correlation) and **pct_negative moves in the opposite direction**. That confirms that the VADER text score and the star rating describe the same thing.
- **Cost has a moderate positive correlation with avg_rating (about 0.4)**, and also with avg_review_words and avg_pictures: pricier places get longer reviews with more pictures.
- **avg_pictures and avg_review_words are strongly related**: customers who write a lot also add pictures.
- n_cuisines and n_collections are only moderately related to the rest.
- ✏️ Features with correlation above 0.85 would be redundant for clustering. The notebook checks this automatically in the feature-selection step.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code (multivariate)
pair_cols = ["Cost", "avg_rating", "avg_sentiment", "avg_pictures", "avg_review_words"]
g = sns.pairplot(restaurants[pair_cols + ["cost_bucket"]], hue="cost_bucket", corner=True, plot_kws={"alpha": 0.7})
g.fig.suptitle("Pair plot of restaurant features, coloured by cost bucket", y=1.02)
plt.show()

##### 1. Why did you pick the specific chart?

A pair plot draws every pair of important numeric features as a scatter plot, with the distribution of each feature on the diagonal, and colours them by cost bucket. It is a quick way to see relationships, clusters and outliers across many variables together.

##### 2. What is/are the insight(s) found from the chart?

- **Premium restaurants (green, cost above Rs. 1,000)** sit mostly at the upper right: higher average rating and higher sentiment than budget places.
- **Budget restaurants (blue)** cover a wide range of ratings (roughly 2.4 to 4.0) but have clearly lower sentiment scores, fewer pictures and shorter reviews, so they form a separate group on those features.
- **avg_pictures and avg_review_words** rise together in a clear upward trend, and **avg_rating and avg_sentiment** form an upward line.
- Cost alone does not guarantee quality: some premium places have ratings of only about 3.2-3.5.
- A couple of very expensive restaurants (around Rs. 2,800) lie far to the right of everything else. These are the outliers that are capped later.
- The three colours overlap, which warns us that segments will be *overlapping* and not perfectly separated.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Based on the chart experiments, three statements are tested:
1. **Cost and rating:** more expensive restaurants have higher average ratings (Chart 7).
2. **Pictures and rating:** reviews that include pictures give higher ratings than reviews without pictures (Chart 10).
3. **Critics and rating:** critics (10+ reviews and 500+ followers) give different ratings from other reviewers (Chart 12).

All tests use a significance level of 5% (alpha = 0.05).

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

- **H0 (null):** There is no monotonic relationship between the cost for one person and the restaurant's average rating (rho = 0).
- **H1 (alternative):** There is a relationship between cost and average rating (rho is not 0).

#### 2. Perform an appropriate statistical test.

In [ ]:
# Hypothesis 1: Cost vs average rating (restaurant level) -> Spearman rank correlation
ALPHA = 0.05
rho, p_value = spearmanr(restaurants["Cost"], restaurants["avg_rating"])
print(f"Spearman rho = {rho:.3f} | p-value = {p_value:.5f}")
print("Decision:", "REJECT H0 (significant relationship)" if p_value < ALPHA else "FAIL TO REJECT H0 (no significant relationship)")

##### Which statistical test have you done to obtain P-Value?

**Spearman rank correlation test** (restaurant level, 100 restaurants).

##### Why did you choose the specific statistical test?

Cost and average rating are both numeric, but cost is right-skewed and the relationship does not have to be a straight line. Spearman uses ranks, so it needs no normality assumption and is not distorted by extreme values. **Result:** rho is about 0.38 and p is about 0.0001, which is far below 0.05, so **we reject H0**: more expensive restaurants tend to have higher ratings (a moderate relationship, not a strong one).

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

- **H0 (null):** Reviews with pictures and reviews without pictures have the same rating distribution.
- **H1 (alternative):** The rating distributions of the two groups differ.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Hypothesis 2: reviews with pictures vs without -> Mann-Whitney U test
with_pic = reviews.loc[reviews["has_picture"], "Rating"]
no_pic = reviews.loc[~reviews["has_picture"], "Rating"]
stat, p_value = mannwhitneyu(with_pic, no_pic, alternative="two-sided")
print(f"Mean rating WITH pictures: {with_pic.mean():.2f} (n={len(with_pic)}) | WITHOUT: {no_pic.mean():.2f} (n={len(no_pic)})")
print(f"U = {stat:.0f} | p-value = {p_value:.5f}")
print("Decision:", "REJECT H0 (ratings differ)" if p_value < ALPHA else "FAIL TO REJECT H0 (no significant difference)")

##### Which statistical test have you done to obtain P-Value?

**Mann-Whitney U test** (two independent groups: 1,983 reviews with pictures, 7,978 without).

##### Why did you choose the specific statistical test?

Ratings are ordinal (1 to 5 stars) and strongly skewed, so a t-test (which assumes roughly normal data) is not appropriate. Mann-Whitney U compares two independent groups without that assumption. **Result:** mean rating 3.86 with pictures versus 3.54 without, p is about 0.003 (below 0.05), so **we reject H0**. Reviews with pictures are rated higher. This is a difference, not proof of cause: happier customers may simply post more pictures.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

- **H0 (null):** Critics and other reviewers give the same rating distribution.
- **H1 (alternative):** Critics give different ratings from other reviewers.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Hypothesis 3: critics vs other reviewers -> Mann-Whitney U test
critic_r = reviews.loc[reviews["is_critic"], "Rating"]
other_r = reviews.loc[~reviews["is_critic"], "Rating"]
stat, p_value = mannwhitneyu(critic_r, other_r, alternative="two-sided")
print(f"Mean rating CRITICS: {critic_r.mean():.2f} (n={len(critic_r)}) | OTHERS: {other_r.mean():.2f} (n={len(other_r)})")
print(f"U = {stat:.0f} | p-value = {p_value:.5f}")
print("Decision:", "REJECT H0 (ratings differ)" if p_value < ALPHA else "FAIL TO REJECT H0 (no significant difference)")

##### Which statistical test have you done to obtain P-Value?

**Mann-Whitney U test** (759 critic reviews versus 9,202 reviews from other reviewers).

##### Why did you choose the specific statistical test?

Same reasons as before: ordinal, skewed ratings and two independent groups of very different size. **Result:** mean 3.80 for critics versus 3.58 for others, but p is about 0.52 (above 0.05), so **we fail to reject H0**. There is not enough evidence that critics rate differently from other customers. The small difference in averages could easily be due to chance.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
print("Missing values left in the analysis tables (before this step):")
print("  reviews    :", int(reviews[["Rating", "Time", "Pictures"]].isna().sum().sum()), "(rating/time/pictures)")
print("  restaurants:", restaurants.isna().sum()[lambda s: s > 0].to_dict())

# 1) Timings: one restaurant has no opening hours -> explicit label
restaurants["Timings"] = restaurants["Timings"].fillna("Not available")

# 2) Numeric review-based features: median imputation (robust to outliers) for any remaining gap
num_feature_cols = restaurants.select_dtypes("number").columns
restaurants[num_feature_cols] = restaurants[num_feature_cols].fillna(restaurants[num_feature_cols].median())

print("\nMissing values after imputation:", int(restaurants.drop(columns=["cost_bucket"]).isna().sum().sum()))

#### What all missing value imputation techniques have you used and why did you use those techniques?

- **Reviews:** rows without a rating (38 almost-empty rows plus the "Like" row) were removed because without a rating they cannot be used; missing review text was set to an empty string and only rows with text are used for NLP.
- **Collections:** missing = "no collection tag", so it was set to an empty string (and later turned into the count n_collections and a flag).
- **Timings:** one missing value was filled with the label "Not available" (it is not used for modelling).
- **Numeric features:** a **median** imputation is used as a safety net, because the median is not affected by outliers.

I avoided deleting restaurants: with only about 100 restaurants every row matters.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
BASE_NUM = ["Cost", "avg_rating", "avg_sentiment", "pct_negative", "avg_pictures",
            "avg_review_words", "avg_reviewer_followers", "n_cuisines", "n_collections"]

# IQR rule: anything beyond 1.5 x IQR from the quartiles is an outlier
cap_bounds, rows = {}, []
for c in BASE_NUM:
    q1, q3 = restaurants[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    cap_bounds[c] = (lo, hi)
    rows.append((c, int(((restaurants[c] < lo) | (restaurants[c] > hi)).sum()), round(lo, 2), round(hi, 2)))
display(pd.DataFrame(rows, columns=["feature", "n_outliers", "lower_bound", "upper_bound"]))

# Treatment: cap (winsorize) instead of deleting - there are only ~100 restaurants, every row matters
restaurants_capped = restaurants.copy()
for c in BASE_NUM:
    restaurants_capped[c] = restaurants_capped[c].clip(*cap_bounds[c])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=restaurants[["avg_reviewer_followers"]], orient="h", ax=axes[0]); axes[0].set_title("avg_reviewer_followers - before capping")
sns.boxplot(data=restaurants_capped[["avg_reviewer_followers"]], orient="h", ax=axes[1]); axes[1].set_title("after capping")
plt.tight_layout(); plt.show()

##### What all outlier treatment techniques have you used and why did you use those techniques?

The **IQR rule** (values beyond 1.5 x IQR from the quartiles) was used to detect outliers in the restaurant features. For example, Cost has 2 restaurants above Rs. 2,250 and n_collections has 7 unusually high values. The outliers were **capped (winsorised)** to the IQR limits and not removed, because (1) there are only about 100 restaurants, (2) the extreme values are real (luxury hotels), and (3) K-Means and hierarchical clustering are sensitive to extreme values, so capping stops a few restaurants from deciding the clusters. The cap limits are stored so that new restaurants can be treated in the same way.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
# 1) Cuisines: a restaurant serves several cuisines -> multi-hot (0/1) columns for the 8 most common cuisines
TOP_N_CUISINES = 8
top_cuisines = list(cuisine_counts.head(TOP_N_CUISINES).index)
for c in top_cuisines:
    restaurants_capped["cuis_" + c.replace(" ", "_")] = restaurants_capped["cuisine_list"].apply(lambda lst, c=c: int(c in lst))

# 2) Collections: only whether the restaurant has any collection tag is kept as a flag
restaurants_capped["has_collection"] = (restaurants_capped["n_collections"] > 0).astype(int)

# 3) Cost bucket is ordered (Budget < Mid < Premium) -> ordinal codes 0/1/2
restaurants_capped["cost_bucket_code"] = restaurants_capped["cost_bucket"].cat.codes

# 4) Review target for the sentiment classifier: positive = 1, negative = 0 (created in the text section)
print("Cuisine flags created:", [c for c in restaurants_capped.columns if c.startswith("cuis_")])
display(restaurants_capped.filter(regex="^(Name|cuis_|has_collection|cost_bucket_code)").head())

#### What all categorical encoding techniques have you used & why did you use those techniques?

- **Multi-hot encoding** for *Cuisines*: one restaurant serves several cuisines, so one-hot encoding is not enough. The 8 most common cuisines became 0/1 columns (North Indian, Chinese, Continental, Biryani, Asian, Italian, Desserts, Fast Food). Rare cuisines were left out to avoid dozens of almost-empty columns. These flags are used to *describe* the clusters.
- **Binary flag** for *Collections* (has_collection).
- **Ordinal encoding** for the cost bucket (Budget < Mid < Premium), because the order has a meaning.
- **Label encoding** for the review target (positive = 1, negative = 0).

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand Contraction
# Text dataset for the classifier: clearly positive (>= 4 stars) vs clearly negative (<= 2.5 stars). 3-star reviews are ambiguous.
text_df = reviews_text[(reviews_text["Rating"] >= 4) | (reviews_text["Rating"] <= 2.5)][["Review", "Rating"]].copy()
text_df["target"] = (text_df["Rating"] >= 4).astype(int)       # 1 = positive, 0 = negative

def expand_contractions(text):
    text = text.replace("\u2019", "'")                           # curly apostrophe -> straight
    text = re.sub(r"\bcan't\b", "cannot", text, flags=re.I)
    text = re.sub(r"\bwon't\b", "will not", text, flags=re.I)
    text = re.sub(r"n't\b", " not", text, flags=re.I)
    text = re.sub(r"'re\b", " are", text, flags=re.I)
    text = re.sub(r"'ve\b", " have", text, flags=re.I)
    text = re.sub(r"'ll\b", " will", text, flags=re.I)
    text = re.sub(r"'d\b", " would", text, flags=re.I)
    text = re.sub(r"'m\b", " am", text, flags=re.I)
    text = re.sub(r"'s\b", "", text)                             # possessive / "is" -> dropped
    return text

text_df["clean_text"] = text_df["Review"].apply(expand_contractions)
print("Rows for the classifier:", len(text_df))
print("BEFORE:", "I didn't like it, they won't refund & it's cold")
print("AFTER :", expand_contractions("I didn't like it, they won't refund & it's cold"))

#### 2. Lower Casing

In [ ]:
# Lower Casing
text_df["clean_text"] = text_df["clean_text"].str.lower()
print(text_df["clean_text"].iloc[0][:150])

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations
# Tokens that start with http/www are kept untouched here; the next step removes them as whole URLs.
def remove_punctuation(text):
    tokens = []
    for tok in text.split():
        tokens.append(tok if tok.startswith(("http", "www")) else re.sub(r"[^\w\s]", " ", tok))
    return " ".join(tokens)

text_df["clean_text"] = text_df["clean_text"].apply(remove_punctuation)
print(text_df["clean_text"].iloc[0][:150])

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits
def remove_urls_and_digit_words(text):
    text = re.sub(r"(https?://\S+|www\.\S+)", " ", text)         # URLs
    text = re.sub(r"\b\w*\d\w*\b", " ", text)                    # any word that contains a digit
    return text

text_df["clean_text"] = text_df["clean_text"].apply(remove_urls_and_digit_words)
print("Example:", remove_urls_and_digit_words("visit www.abc.com or call 9999 food4u was great"))

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords
# sklearn's stop-word list, but WITHOUT negations/intensifiers - "not good" must stay different from "good"
KEEP = {"not", "no", "nor", "never", "cannot", "nothing", "none", "very", "too"}
STOP_WORDS = set(ENGLISH_STOP_WORDS) - KEEP

def remove_stopwords(text):
    return " ".join(w for w in text.split() if w not in STOP_WORDS)

text_df["clean_text"] = text_df["clean_text"].apply(remove_stopwords)
print(text_df["clean_text"].iloc[0][:150])

In [ ]:
# Remove White spaces
text_df["clean_text"] = text_df["clean_text"].apply(lambda t: " ".join(t.split()))
print(repr(text_df["clean_text"].iloc[0][:150]))

#### 6. Rephrase Text

In [ ]:
# Rephrase Text
# Reviews contain slang and stretched words ("sooooo gud"). We map common slang and shorten repeated letters.
SLANG = {"thnx": "thanks", "thx": "thanks", "tnx": "thanks", "pls": "please", "plz": "please",
         "gud": "good", "awsm": "awesome", "ok": "okay", "resto": "restaurant", "ambiance": "ambience"}

def rephrase(text):
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)                    # sooooo -> soo
    return " ".join(SLANG.get(w, w) for w in text.split())

text_df["clean_text"] = text_df["clean_text"].apply(rephrase)
print(rephrase("sooooo gud thnx"))

#### 7. Tokenization

In [ ]:
# Tokenization
# After the cleaning above, splitting on spaces is a safe and fast tokenizer.
text_df["tokens"] = text_df["clean_text"].str.split()
print(text_df["tokens"].iloc[0][:15])

#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)
def build_lemmatizer():
    """WordNet lemmatizer if NLTK + corpus are available, otherwise a very light plural-stripping fallback."""
    try:
        lem = WordNetLemmatizer()
        lem.lemmatize("tests")                                    # forces the corpus to load; raises if missing
        return lem.lemmatize
    except Exception:
        def simple(w):
            if w.endswith("ies") and len(w) > 4:
                return w[:-3] + "y"
            if w.endswith("s") and not w.endswith("ss") and len(w) > 3:
                return w[:-1]
            return w
        return simple

lemmatize = build_lemmatizer()
text_df["lemma_text"] = text_df["tokens"].apply(lambda toks: " ".join(lemmatize(t) for t in toks))

# Some very short reviews (e.g. "Ok!") become empty after cleaning -> drop them
text_df = text_df[text_df["lemma_text"].str.len() > 0].reset_index(drop=True)
print("Rows left after cleaning:", len(text_df))
display(text_df[["Review", "lemma_text"]].head(3))

def preprocess_review(text):
    """The complete cleaning chain in one function - used later to score brand-new, unseen reviews."""
    text = expand_contractions(text).lower()
    text = remove_punctuation(text)
    text = remove_urls_and_digit_words(text)
    text = remove_stopwords(text)
    text = rephrase(" ".join(text.split()))
    return " ".join(lemmatize(w) for w in text.split())

##### Which text normalization technique have you used and why?

**Lemmatization** (WordNet lemmatizer, with a simple plural-stripping fallback if NLTK is not available). Lemmatization turns words into real dictionary forms (e.g. 'dishes' -> 'dish', 'ordered' -> 'order'), so different forms of a word count as one feature. I preferred it to stemming because stemming cuts words into non-words ('delicious' -> 'delici'), which makes the results harder to explain.

#### 9. Part of speech tagging

In [ ]:
# POS Taging
# NLTK's tagger is used on one example review (skipped gracefully if the tagger is not installed).
try:
    sample_tokens = text_df["tokens"].iloc[0][:12]
    print(nltk.pos_tag(sample_tokens))
except Exception as err:
    print("POS tagging skipped (NLTK tagger not available):", type(err).__name__)

#### 10. Text Vectorization

In [ ]:
# Vectorizing Text
# TF-IDF with unigrams + bigrams. This cell only shows what the vectors look like;
# the vectorizer used for modelling is fitted INSIDE the model pipeline on training data only (no data leakage).
tfidf_demo = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=20000)
X_demo = tfidf_demo.fit_transform(text_df["lemma_text"])
print("TF-IDF matrix:", X_demo.shape, "(reviews x terms)")
idf = pd.Series(tfidf_demo.idf_, index=tfidf_demo.get_feature_names_out()).sort_values()
print("Most common (lowest idf):", list(idf.index[:10]))
print("Rarest (highest idf)    :", list(idf.index[-5:]))

##### Which text vectorization technique have you used and why?

**TF-IDF (unigrams + bigrams)**. Unlike a plain word count, TF-IDF reduces the weight of words that appear in almost every review (like 'food' or 'good') and increases the weight of distinctive words ('rude', 'undercooked'). Bigrams catch short phrases such as 'not good' or 'highly recommend'. min_df = 3 removes very rare words and max_features = 20,000 limits the size. To avoid data leakage, the final vectorizer is fitted inside the model pipeline on the training data only.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
def add_new_features(df):
    """New features (also reused later when scoring an unseen restaurant)."""
    df = df.copy()
    df["value_for_money"] = df["avg_rating"] / (df["Cost"] / 100)     # rating points per Rs.100 spent
    df["has_collection"] = (df["n_collections"] > 0).astype(int)
    return df

restaurants_capped = add_new_features(restaurants_capped)

CANDIDATES = ["Cost", "avg_rating", "avg_sentiment", "pct_negative", "avg_pictures",
              "avg_review_words", "avg_reviewer_followers", "n_cuisines", "n_collections", "value_for_money"]

# Highly correlated features carry the same information twice and over-weight it in distance-based clustering
corr = restaurants_capped[CANDIDATES].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if (upper[c] > 0.85).any()]
print("Features dropped because |correlation| > 0.85 with another feature:", to_drop or "none")

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting
FEATURES = [c for c in CANDIDATES if c not in to_drop]

# near-constant features add noise only
variances = restaurants_capped[FEATURES].var()
print("Features with ~zero variance:", list(variances[variances < 1e-8].index) or "none")
print("\nFinal clustering features (", len(FEATURES), "):", FEATURES)

##### What all feature selection methods have you used  and why?

- **Correlation filter:** features with an absolute correlation above 0.85 with another feature are dropped, because they would count the same information twice in distance-based clustering. ✏️ (see the printout of the notebook for which, if any, were dropped).
- **Variance check:** features with almost no variance would add noise only.
- **Domain choice:** I kept only features that describe price, quality, customer sentiment, engagement and variety of a restaurant. Identifiers (name, link, timings) were excluded.
- For the text classifier, feature selection is handled by TF-IDF (min_df, max_features) and by the regularisation of the models.

##### Which all features you found important and why?

The most useful features are the ones that separate restaurants in a business-relevant way: **Cost** (price tier), **avg_rating** and **avg_sentiment** (quality and mood), **pct_negative** (risk), **avg_pictures** and **avg_review_words** (customer engagement), and **value_for_money** (rating per Rs. 100 spent, a new feature). The Random Forest feature-importance chart in the explainability section shows which of them separate the final clusters the most.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

**Answer:** Yes, partly. Strongly right-skewed non-negative features (skewness above 1) were transformed with **log(1 + x)**, which pulls in the long tail so that a few restaurants do not dominate the distances. The notebook tests this automatically and prints the skewness before and after (typically avg_pictures and value_for_money). Features that are already roughly symmetric were left as they are.

In [ ]:
# Transform Your data
# Log transform for strongly right-skewed, non-negative features (skewness > 1)
X_model = restaurants_capped[FEATURES].copy()
skew_before = X_model.skew()
LOG_COLS = [c for c in FEATURES if skew_before[c] > 1 and X_model[c].min() >= 0]
X_model[LOG_COLS] = np.log1p(X_model[LOG_COLS])

print("Log-transformed columns:", LOG_COLS or "none")
display(pd.DataFrame({"skew before": skew_before, "skew after": X_model.skew()}).round(2))

### 6. Data Scaling

In [ ]:
# Scaling your data
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_model), columns=FEATURES, index=X_model.index)
print(X_scaled.describe().loc[["mean", "std"]].round(2))

##### Which method have you used to scale you data and why?

**Answer:** **StandardScaler** (z-score: mean 0, standard deviation 1). K-Means, hierarchical clustering and Gaussian Mixtures all depend on distances. Without scaling, Cost (in hundreds of rupees) would dominate features such as the 1-5 rating. StandardScaler was chosen over Min-Max scaling because Min-Max is very sensitive to the extreme values that remain after capping.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

**Not needed for the models.** There are only about 10 features and 100 restaurants, so there is no real risk of the 'curse of dimensionality', and clustering on the original features keeps the clusters easy to explain (each axis has a business meaning). PCA is only used to **visualise** the clusters in 2D.

In [ ]:
# DImensionality Reduction (If needed)
# PCA is used to LOOK at the clusters in 2D (not to train them).
pca_full = PCA(random_state=RANDOM_STATE).fit(X_scaled)
cum = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cum) + 1), cum, "o-")
plt.axhline(0.8, color="red", ls="--", label="80% variance")
plt.xlabel("Number of principal components"); plt.ylabel("Cumulative explained variance")
plt.title("PCA - how many components are needed?"); plt.legend(); plt.show()

pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
coords_2d = pca_2d.fit_transform(X_scaled)
print(f"First 2 components explain {pca_2d.explained_variance_ratio_.sum() * 100:.0f}% of the variance")

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

**PCA** (Principal Component Analysis). It compresses the scaled features into two components for plotting. The cumulative-variance chart shows how much information is kept (the first two components explain roughly 60% of the variance), so the 2D picture is useful for looking at the clusters but is a simplified view.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
# (Used for the supervised review-sentiment classifier; the clustering models use all restaurants.)
X_train, X_test, y_train, y_test = train_test_split(
    text_df["lemma_text"], text_df["target"],
    test_size=0.2, stratify=text_df["target"], random_state=RANDOM_STATE)
print("Train:", len(X_train), "| Test:", len(X_test))
print("Positive share - train: %.1f%% | test: %.1f%%" % (y_train.mean() * 100, y_test.mean() * 100))

##### What data splitting ratio have you used and why?

**80% training / 20% testing, stratified** (the positive/negative ratio is the same in both). With about 8,700 labelled reviews, 20% gives a test set of about 1,740 reviews, which is large enough for a reliable score, while 80% leaves plenty of data to learn from. The test set is only used once, at the end. 5-fold cross-validation is used inside the training set for model selection and tuning. The clustering models do not need a split, because there is no target to predict.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

**Yes, for the sentiment classifier.** About 72% of the labelled reviews are positive and only about 28% are negative (roughly 2.6 : 1). A model that always says 'positive' would already be about 72% accurate, so accuracy alone would be misleading.

In [ ]:
# Handling Imbalanced Dataset (If needed)
counts = y_train.value_counts().sort_index()
print("Training class counts (0 = negative, 1 = positive):", counts.to_dict())

# Balanced class weights = n_samples / (n_classes * class_count); passed to the models as class_weight="balanced"
weights = len(y_train) / (2 * counts)
print("Class weights used by class_weight='balanced':", weights.round(2).to_dict())

plt.figure(figsize=(5, 4))
sns.barplot(x=["Negative", "Positive"], y=counts.values, color="#4C72B0")
plt.title("Class balance of the training set"); plt.ylabel("Reviews"); plt.show()

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

**Class weights** (`class_weight='balanced'`) in Logistic Regression and Linear SVM: mistakes on the minority (negative) class cost more during training. I chose it over oversampling (e.g. SMOTE) because SMOTE creates synthetic points in a sparse TF-IDF space, which makes little sense for text, and over undersampling because that would throw away a lot of good data. I also judge the models with macro-F1, the recall of the negative class and the confusion matrix, not with accuracy alone.

## ***7. ML Model Implementation***

### ML Model - 1 : K-Means Clustering

In [ ]:
# ML Model - 1 Implementation : K-Means

# ---- helper functions reused by all three clustering models ----
def evaluate_clusters(X, labels):
    """Three standard internal metrics (no true labels exist in unsupervised learning)."""
    return {"Silhouette": silhouette_score(X, labels),
            "Davies-Bouldin": davies_bouldin_score(X, labels),
            "Calinski-Harabasz": calinski_harabasz_score(X, labels)}

def stability_score(make_model, X, n_trials=10, frac=0.8, seed=RANDOM_STATE):
    """Cross-validation idea for clustering: refit on random 80% subsamples and check that the
    groupings agree with the full-data clustering (Adjusted Rand Index; 1 = identical, 0 = random)."""
    rng = np.random.RandomState(seed)
    ref = make_model().fit_predict(X)
    scores = []
    for _ in range(n_trials):
        idx = rng.choice(len(X), int(frac * len(X)), replace=False)
        sub = make_model().fit_predict(X[idx])
        scores.append(adjusted_rand_score(ref[idx], sub))
    return float(np.mean(scores))

def plot_scores(score_dict, title):
    """Bar chart of the three metrics for several models / versions."""
    metrics = ["Silhouette", "Davies-Bouldin", "Calinski-Harabasz"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, m in zip(axes, metrics):
        names = list(score_dict.keys())
        vals = [score_dict[n][m] for n in names]
        ax.bar(names, vals, color=sns.color_palette()[:len(names)])
        ax.set_title(m + (" (lower = better)" if m == "Davies-Bouldin" else " (higher = better)"))
        ax.tick_params(axis="x", rotation=15)
        for i, v in enumerate(vals):
            ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")
    fig.suptitle(title, y=1.03); plt.tight_layout(); plt.show()

X_arr = X_scaled.values

# ---- baseline model: K-Means with k = 3 and default settings ----
kmeans_base = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE)
labels_km_base = kmeans_base.fit_predict(X_arr)
scores_km_base = evaluate_clusters(X_arr, labels_km_base)
print("Baseline K-Means (k=3):", {k: round(v, 3) for k, v in scores_km_base.items()})
print("Cluster sizes:", np.bincount(labels_km_base))

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

**K-Means** splits the restaurants into *k* groups by repeatedly assigning each restaurant to the closest group centre and moving the centres to the mean of their members. It is fast, easy to explain and can score new restaurants. There is no 'true label' in unsupervised learning, so three internal metrics are used:
- **Silhouette** (-1 to 1, higher is better): how close a restaurant is to its own cluster compared with the nearest other cluster.
- **Davies-Bouldin** (lower is better): the average similarity between each cluster and its most similar one.
- **Calinski-Harabasz** (higher is better): the ratio of between-cluster spread to within-cluster spread.

The baseline is K-Means with k = 3. The chart below shows its scores. Silhouette values around 0.2-0.3 mean *moderate* structure: the segments overlap somewhat, which is normal for real business data.

In [ ]:
# Visualizing evaluation Metric Score chart
plot_scores({"K-Means baseline (k=3)": scores_km_base}, "ML Model 1 - K-Means: evaluation metrics")

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (Grid Search over k)
rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    lab = km.fit_predict(X_arr)
    stab = stability_score(lambda k=k: KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE), X_arr)
    rows.append({"k": k, "inertia": km.inertia_, **evaluate_clusters(X_arr, lab), "stability_ARI": stab})
km_results = pd.DataFrame(rows)
display(km_results.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(km_results["k"], km_results["inertia"], "o-"); axes[0].set_title("Elbow (inertia)")
axes[1].plot(km_results["k"], km_results["Silhouette"], "o-", color="darkorange"); axes[1].set_title("Silhouette")
axes[2].plot(km_results["k"], km_results["stability_ARI"], "o-", color="green"); axes[2].set_title("Stability (ARI on 80% resamples)")
for ax in axes: ax.set_xlabel("k")
plt.tight_layout(); plt.show()

# Choose k: look at 3-6 clusters (2 is too coarse to act on), find the best silhouette,
# then take the SMALLEST k within 0.01 of it -> the simplest segmentation that is almost as good.
cand = km_results[(km_results["k"] >= 3) & (km_results["k"] <= 6)]
best_sil = cand["Silhouette"].max()
K_FINAL = int(cand[cand["Silhouette"] >= best_sil - 0.01]["k"].min())
print("Chosen number of clusters K_FINAL =", K_FINAL)

kmeans_final = KMeans(n_clusters=K_FINAL, n_init=20, random_state=RANDOM_STATE).fit(X_arr)
labels_km = kmeans_final.labels_
scores_km_tuned = evaluate_clusters(X_arr, labels_km)
plot_scores({"Baseline (k=3)": scores_km_base, f"Tuned (k={K_FINAL})": scores_km_tuned}, "K-Means: before vs after tuning")

##### Which hyperparameter optimization technique have you used and why?

**Grid Search over k = 2 to 10** (with n_init = 20 restarts), judged by the silhouette score, the elbow curve and a **stability test** (re-clustering random 80% subsets and measuring agreement with the full clustering by the Adjusted Rand Index). For clustering there is no train/validation split to cross-validate on, so the repeated-resampling stability test plays the role of cross-validation. I chose the grid because k is the one hyperparameter that really matters for K-Means, and it is cheap to search.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

✏️ Compare the 'Baseline (k=3)' and 'Tuned' bars in your chart and write the exact numbers. In my test run the tuned model ended up with the same k as the baseline, so the scores were almost identical; expect a small change at most. That is still a valid result: the grid search **confirms** that the chosen k is a good one (compare its silhouette and stability with the other k values in the table), and the extra restarts (n_init = 20) guard against a bad starting point. If your run selected a different k, describe the change in silhouette, Davies-Bouldin and stability here.

### ML Model - 2 : Agglomerative (Hierarchical) Clustering

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

**Agglomerative (hierarchical) clustering with Ward linkage** starts with every restaurant as its own cluster and repeatedly merges the two clusters whose merge increases the within-cluster variance the least. It produces a **dendrogram** (a tree of merges), which helps to see how many natural groups exist. It has no 'predict' for a new restaurant. The baseline here is Ward linkage with k = 3, evaluated with the same three metrics.

In [ ]:
# ML Model - 2 Implementation : Agglomerative (hierarchical) clustering + evaluation metric chart
agg_base = AgglomerativeClustering(n_clusters=3, linkage="ward")
labels_agg_base = agg_base.fit_predict(X_arr)
scores_agg_base = evaluate_clusters(X_arr, labels_agg_base)
print("Baseline Agglomerative (ward, k=3):", {k: round(v, 3) for k, v in scores_agg_base.items()})

# Dendrogram: the tree of merges. Cutting it at a height gives a number of clusters.
Z = linkage(X_arr, method="ward")
plt.figure(figsize=(13, 4.5))
dendrogram(Z, truncate_mode="lastp", p=20, leaf_rotation=90, show_contracted=True)
plt.title("Dendrogram (Ward linkage, last 20 merges)"); plt.xlabel("Restaurants (or groups)"); plt.ylabel("Distance")
plt.show()

plot_scores({"Agglomerative baseline (k=3)": scores_agg_base}, "ML Model 2 - Agglomerative: evaluation metrics")

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 2 Implementation with hyperparameter optimization (Grid Search over linkage and k)
rows = []
for link in ["ward", "complete", "average"]:
    for k in range(3, 9):
        lab = AgglomerativeClustering(n_clusters=k, linkage=link).fit_predict(X_arr)
        rows.append({"linkage": link, "k": k, **evaluate_clusters(X_arr, lab)})
agg_results = pd.DataFrame(rows)
display(agg_results.sort_values("Silhouette", ascending=False).head(8).round(3))

best = agg_results.sort_values("Silhouette", ascending=False).iloc[0]
best_link, best_k_agg = best["linkage"], int(best["k"])
print(f"Best settings by silhouette: linkage = {best_link}, k = {best_k_agg}")

labels_agg = AgglomerativeClustering(n_clusters=best_k_agg, linkage=best_link).fit_predict(X_arr)
scores_agg_tuned = evaluate_clusters(X_arr, labels_agg)
stab_agg = stability_score(lambda: AgglomerativeClustering(n_clusters=best_k_agg, linkage=best_link), X_arr)
print("Cluster sizes:", np.bincount(labels_agg), "| stability (ARI):", round(stab_agg, 3))
plot_scores({"Baseline (ward, k=3)": scores_agg_base, f"Tuned ({best_link}, k={best_k_agg})": scores_agg_tuned},
            "Agglomerative: before vs after tuning")

##### Which hyperparameter optimization technique have you used and why?

**Grid Search over the linkage type (ward, complete, average) and k = 3 to 8**, scored by the silhouette. The linkage decides how the distance between two clusters is measured, and it can change the result a lot, so it is the main hyperparameter to search. A resampling stability score (ARI) was also calculated for the best settings.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

✏️ Compare the 'Baseline' and 'Tuned' bars and write the exact numbers. In my test run Ward linkage with k = 3 was already the best combination, so there was little or no improvement; check whether that is true for you too. Note the pattern in the table: 'average' linkage can look better on Davies-Bouldin but tends to create tiny clusters, and the stability (ARI) of the hierarchical model is usually lower than K-Means, which means its groups change more when some restaurants are removed.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

**What each metric means for the business**
- **Silhouette (higher is better):** how clearly a restaurant belongs to its segment. A low value means that some restaurants sit between two segments, so a marketing rule for a segment will not fit all its members. Around 0.25 means 'useful but overlapping' segments.
- **Davies-Bouldin (lower is better):** whether the segments are different from each other. If it is high, two segments are similar and one strategy could serve both.
- **Calinski-Harabasz (higher is better):** how compact and separate the groups are overall.
- **Stability (ARI):** whether the segments stay the same if the data changes a little. Unstable segments are dangerous to build strategies on.

**Business impact:** good segments let Zomato (1) promote 'hidden gems', (2) support under-performing restaurants, (3) personalise recommendations by price and quality, and (4) allocate marketing budgets by segment instead of treating all restaurants alike.

### ML Model - 3 : Gaussian Mixture Model (GMM)

In [ ]:
# ML Model - 3 Implementation : Gaussian Mixture Model (soft clustering)
gmm_base = GaussianMixture(n_components=3, random_state=RANDOM_STATE)
labels_gmm_base = gmm_base.fit_predict(X_arr)
scores_gmm_base = evaluate_clusters(X_arr, labels_gmm_base)
print("Baseline GMM (k=3, full covariance):", {k: round(v, 3) for k, v in scores_gmm_base.items()})
print("Cluster sizes:", np.bincount(labels_gmm_base))

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

**Gaussian Mixture Model (GMM)** assumes the data comes from several bell-shaped (Gaussian) groups and estimates the shape, size and position of each group. Unlike K-Means it gives *soft* assignments (the probability that a restaurant belongs to each group) and can model groups of different shapes and sizes. The baseline is GMM with k = 3 and full covariance.

In [ ]:
# Visualizing evaluation Metric Score chart
plot_scores({"GMM baseline (k=3)": scores_gmm_base}, "ML Model 3 - Gaussian Mixture: evaluation metrics")

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization (Grid Search over k and covariance type)
rows = []
for cov in ["full", "tied", "diag", "spherical"]:
    for k in range(3, 9):
        try:
            gm = GaussianMixture(n_components=k, covariance_type=cov, n_init=3, random_state=RANDOM_STATE)
            lab = gm.fit_predict(X_arr)
            if len(set(lab)) < 2:
                continue
            rows.append({"covariance": cov, "k": k, "BIC": gm.bic(X_arr), **evaluate_clusters(X_arr, lab)})
        except Exception as err:
            print("skipped", cov, k, type(err).__name__)
gmm_results = pd.DataFrame(rows)
display(gmm_results.sort_values("Silhouette", ascending=False).head(8).round(3))

best = gmm_results.sort_values("Silhouette", ascending=False).iloc[0]
best_cov, best_k_gmm = best["covariance"], int(best["k"])
print(f"Best settings by silhouette: covariance = {best_cov}, k = {best_k_gmm}")

gmm_final = GaussianMixture(n_components=best_k_gmm, covariance_type=best_cov, n_init=3, random_state=RANDOM_STATE).fit(X_arr)
labels_gmm = gmm_final.predict(X_arr)
scores_gmm_tuned = evaluate_clusters(X_arr, labels_gmm)
stab_gmm = stability_score(lambda: GaussianMixture(n_components=best_k_gmm, covariance_type=best_cov, n_init=3, random_state=RANDOM_STATE), X_arr)
print("Cluster sizes:", np.bincount(labels_gmm), "| stability (ARI):", round(stab_gmm, 3))
plot_scores({"Baseline (full, k=3)": scores_gmm_base, f"Tuned ({best_cov}, k={best_k_gmm})": scores_gmm_tuned},
            "Gaussian Mixture: before vs after tuning")

##### Which hyperparameter optimization technique have you used and why?

**Grid Search over the covariance type (full, tied, diag, spherical) and k = 3 to 8**, scored by the silhouette (the BIC is printed too as a second opinion). With only 100 restaurants, a 'full' covariance has many parameters to estimate and can overfit, while a simpler 'spherical' or 'diag' form is often more stable, so the covariance type is the key hyperparameter. n_init = 3 restarts were used for each setting.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

✏️ Compare the 'Baseline' and 'Tuned' bars and write the exact numbers. In my test run the tuned GMM (a simpler covariance type) scored a little better than the full-covariance baseline. That is the pattern to look for: with only 100 restaurants a simpler model often fits better, because a full covariance has too many parameters to estimate. Report what happened in YOUR chart.

### Comparing the three tuned clustering models
The table and chart below put the tuned models side by side. Differences between the models are small, so the choice is made on a mix of **metrics, stability and usefulness**:
- **K-Means** has the highest stability and the most balanced cluster sizes, is easy to explain to non-technical people, and can score a brand-new restaurant (`predict`). The Agglomerative model cannot.
- **GMM** has a similar or slightly better silhouette, but it is harder to explain and less stable.
- **Agglomerative** is useful for the dendrogram but gives lopsided clusters and the lowest stability.

➡️ **Final clustering model: K-Means.**

In [ ]:
# Comparing the three tuned clustering models side by side
comparison = pd.DataFrame({"K-Means": scores_km_tuned, "Agglomerative": scores_agg_tuned, "Gaussian Mixture": scores_gmm_tuned}).T
comparison["n_clusters"] = [K_FINAL, best_k_agg, best_k_gmm]
comparison["stability_ARI"] = [km_results.loc[km_results["k"] == K_FINAL, "stability_ARI"].iloc[0], stab_agg, stab_gmm]
display(comparison.round(3))
plot_scores({"K-Means": scores_km_tuned, "Agglomerative": scores_agg_tuned, "GMM": scores_gmm_tuned}, "Tuned clustering models compared")

### Final clusters: profiles and what makes them different
The profile table (average of every feature per cluster), the heatmap (how far each cluster is from the overall average, in standard deviations) and the cuisine mix tell us **who is in each cluster**. Green = above average, red = below average.

In [ ]:
# Profile of the final clusters (K-Means)
restaurants_final = restaurants_capped.copy()
restaurants_final["cluster"] = labels_km

profile = restaurants_final.groupby("cluster")[FEATURES].mean().round(2)
profile.insert(0, "n_restaurants", restaurants_final["cluster"].value_counts().sort_index())
display(profile)

# How far is each cluster from the overall average? (in standard deviations)
z = (profile[FEATURES] - restaurants_final[FEATURES].mean()) / restaurants_final[FEATURES].std()
plt.figure(figsize=(11, 0.9 * K_FINAL + 2))
sns.heatmap(z, annot=True, fmt=".1f", cmap="RdYlGn", center=0)
plt.title("Cluster profile vs overall average (standard deviations)"); plt.ylabel("Cluster")
plt.show()

# 2D view of the clusters
plt.figure(figsize=(8, 6))
sns.scatterplot(x=coords_2d[:, 0], y=coords_2d[:, 1], hue=restaurants_final["cluster"], palette="deep", s=80)
plt.title(f"K-Means clusters (k={K_FINAL}) in 2D PCA space")
plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0] * 100:.0f}%)"); plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1] * 100:.0f}%)")
plt.show()

In [ ]:
# Cuisine mix and example restaurants of each cluster
cuis_cols = [c for c in restaurants_final.columns if c.startswith("cuis_")]
share = restaurants_final.groupby("cluster")[cuis_cols].mean().mul(100).round(0)
share.columns = [c.replace("cuis_", "").replace("_", " ") for c in share.columns]
print("% of restaurants in the cluster that serve each cuisine:")
display(share)

for c in sorted(restaurants_final["cluster"].unique()):
    top = restaurants_final[restaurants_final["cluster"] == c].nlargest(5, "avg_rating")["Name"].tolist()
    print(f"Cluster {c}: {', '.join(top)}")

### Name each cluster and write a strategy
✏️ Fill this in from your profile table and heatmap above. **This table is the most important part for your video.** Add or remove rows to match the number of clusters (K).

| Cluster | Suggested name | What defines it (use the numbers) | Strategy for **customers** | Strategy for **Zomato** |
|---|---|---|---|---|
| 0 | | | | |
| 1 | | | | |
| 2 | | | | |

**Examples of how to name them (check them against YOUR numbers first):**
- *Premium, highly rated* (high cost, high rating and sentiment, many cuisines and collection tags) -> promote as special-occasion picks, feature in curated collections.
- *Budget with weak ratings* (low cost, low rating, high share of negative reviews) -> support with quality programmes, lower search priority until ratings improve.
- *Popular mid-range / engagement leaders* (many pictures and long reviews, many reviewer followers) -> use as showcase listings and for sponsored promotions.

### Bonus ML models: predicting whether a review is positive or negative (supervised)
Besides segmenting restaurants, a supervised text model is built to check how well the *words* of a review predict a positive (4-5 stars) or negative (1-2.5 stars) review. Three models are cross-validated: **Logistic Regression**, **Multinomial Naive Bayes** and **Linear SVM**. The best one is tuned with Grid Search and evaluated once on the untouched test set. This model is the one that is saved and used for the 'unseen data' check at the end.

In [ ]:
# Bonus supervised models: 5-fold cross-validation of three text classifiers
# TF-IDF sits INSIDE each pipeline, so it is re-fitted on the training part of every fold (no leakage).
def make_pipeline(clf, **tfidf_kwargs):
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=20000, **tfidf_kwargs)),
                     ("clf", clf)])

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(class_weight="balanced", random_state=RANDOM_STATE),
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
for name, clf in classifiers.items():
    cv = cross_validate(make_pipeline(clf), X_train, y_train, cv=skf,
                        scoring=["accuracy", "precision", "recall", "f1", "f1_macro"])
    cv_rows.append({"model": name, "accuracy": cv["test_accuracy"].mean(), "precision": cv["test_precision"].mean(),
                    "recall": cv["test_recall"].mean(), "f1 (positive)": cv["test_f1"].mean(),
                    "f1 macro": cv["test_f1_macro"].mean(), "f1 macro std": cv["test_f1_macro"].std()})
cv_table = pd.DataFrame(cv_rows).set_index("model")
display(cv_table.round(3))

cv_table[["accuracy", "precision", "recall", "f1 macro"]].plot(kind="bar", figsize=(10, 4.5), rot=0)
plt.ylim(0.5, 1); plt.title("Cross-validated scores of the three text classifiers (training data)"); plt.ylabel("Score")
plt.show()

In [ ]:
# Hyperparameter tuning (Grid Search, 5-fold CV, scored on macro-F1) for the best model of the table above
best_name = cv_table["f1 macro"].idxmax()
print("Best model in cross-validation:", best_name)

param_grids = {
    "Logistic Regression": {"clf__C": [0.5, 1, 2, 5], "tfidf__ngram_range": [(1, 1), (1, 2)]},
    "Naive Bayes":         {"clf__alpha": [0.1, 0.5, 1.0], "tfidf__ngram_range": [(1, 1), (1, 2)]},
    "Linear SVM":          {"clf__C": [0.1, 0.5, 1, 2], "tfidf__ngram_range": [(1, 1), (1, 2)]},
}
grid = GridSearchCV(make_pipeline(classifiers[best_name]), param_grids[best_name],
                    scoring="f1_macro", cv=skf, n_jobs=-1)
grid.fit(X_train, y_train)
print("Best parameters:", grid.best_params_, "| best CV macro-F1: %.3f" % grid.best_score_)

# Compare on the untouched TEST set: baseline vs tuned
baseline_model = make_pipeline(classifiers[best_name]).fit(X_train, y_train)
best_sentiment_model = grid.best_estimator_

def test_scores(model):
    pred = model.predict(X_test)
    return {"Accuracy": accuracy_score(y_test, pred), "Precision": precision_score(y_test, pred),
            "Recall": recall_score(y_test, pred), "F1": f1_score(y_test, pred),
            "Negative-class recall": recall_score(y_test, pred, pos_label=0)}

compare = pd.DataFrame({"Baseline": test_scores(baseline_model), "Tuned": test_scores(best_sentiment_model)}).T
display(compare.round(3))
compare.T.plot(kind="bar", figsize=(10, 4.5), rot=0); plt.ylim(0.5, 1)
plt.title("Test-set scores: baseline vs tuned"); plt.show()

In [ ]:
# Final evaluation of the tuned classifier on the test set
pred = best_sentiment_model.predict(X_test)
print(classification_report(y_test, pred, target_names=["Negative", "Positive"]))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["Negative", "Positive"]).plot(cmap="Blues")
plt.title("Confusion matrix - test set"); plt.show()

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

**For the clustering models** (no true labels exist): **Silhouette** (main metric: how clearly each restaurant belongs to its group), **Davies-Bouldin** and **Calinski-Harabasz** (to check that groups are compact and separate), plus the **stability (ARI)** test (segments must not change when the data changes slightly, otherwise a business cannot rely on them).

**For the review-sentiment classifier:** **Accuracy alone is misleading** because about 72% of reviews are positive. I focus on:
- **Macro-F1** (treats both classes equally) - used for model selection and tuning.
- **Recall of the negative class** - the share of really unhappy customers that the model finds. Missing a negative review means a problem stays hidden, so this matters most to the business.
- **Precision of the negative class** - how many flagged reviews are really negative, so that support teams do not waste time on false alarms.
- The **confusion matrix** to see the exact kinds of mistakes.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

- **Clustering: K-Means.** All three models have similar scores, but K-Means is the most stable, gives balanced and explainable segments and can place new restaurants into a segment, which is needed for deployment.
- **Sentiment classifier: the model with the highest cross-validated macro-F1** (in my run, the Linear SVM or Logistic Regression with TF-IDF unigrams + bigrams, both around 0.94-0.95 macro-F1; Naive Bayes was clearly weaker on negative reviews). ✏️ Check your table and write the name of the model that won in your run. A linear model on TF-IDF is fast, hard to overfit and its weights can be read as words, so it is easy to explain.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

In [ ]:
# Model explainability (1/2): which WORDS drive the sentiment classifier?
tfidf_fitted = best_sentiment_model.named_steps["tfidf"]
clf_fitted = best_sentiment_model.named_steps["clf"]
words = np.array(tfidf_fitted.get_feature_names_out())
if hasattr(clf_fitted, "coef_"):                       # Logistic Regression / Linear SVM
    weights = clf_fitted.coef_[0]
else:                                                   # Naive Bayes
    weights = clf_fitted.feature_log_prob_[1] - clf_fitted.feature_log_prob_[0]

order = np.argsort(weights)
top_neg, top_pos = order[:15], order[-15:]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(words[top_neg], weights[top_neg], color="#C44E52"); axes[0].set_title("Words pushing towards NEGATIVE")
axes[1].barh(words[top_pos], weights[top_pos], color="#55A868"); axes[1].set_title("Words pushing towards POSITIVE")
plt.tight_layout(); plt.show()

In [ ]:
# Model explainability (2/2): which FEATURES define the restaurant clusters?
# Train a Random Forest to predict the cluster label from the features - its importances show what separates the clusters.
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, oob_score=True).fit(X_scaled, labels_km)
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
print("Out-of-bag accuracy of 'cluster from features':", round(rf.oob_score_, 3))
plt.figure(figsize=(8, 5))
imp.plot(kind="barh", color="#4C72B0"); plt.title("Feature importance for separating the clusters"); plt.xlabel("Importance")
plt.show()

**Sentiment model:** TF-IDF turns each review into numbers, and a linear classifier learns a weight for each word. The word charts above show the words with the largest weights: positive words such as *best, amazing, delicious, excellent* push a review towards positive, and words such as *worst, bad, disappointed, rude, poor* push it towards negative. These make sense, which is a good sign that the model learned real signals and not noise. Negations ('not good') are kept as features because stop-word removal did not remove 'not'.

**Cluster model:** a Random Forest was trained to predict the cluster label from the features. Its feature-importance chart shows what separates the segments most (typically cost, value for money, sentiment/rating and engagement features such as pictures and review length). The out-of-bag accuracy tells how well the segments can be explained by the features. (SHAP could also be used here; I used model weights and Random Forest importance because they need no extra installation and are easy to explain.)

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File
# 1) the sentiment classifier (TF-IDF + model, one Pipeline object)
joblib.dump(best_sentiment_model, "zomato_sentiment_model.joblib")

# 2) the restaurant-segmentation model together with everything needed to prepare a new restaurant the same way
cluster_bundle = {"features": FEATURES, "log_cols": LOG_COLS, "cap_bounds": cap_bounds,
                  "scaler": scaler, "model": kmeans_final, "k": K_FINAL}
joblib.dump(cluster_bundle, "zomato_cluster_model.joblib")
print("Saved:", [f for f in os.listdir(".") if f.endswith(".joblib")])

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.
sentiment_model = joblib.load("zomato_sentiment_model.joblib")
bundle = joblib.load("zomato_cluster_model.joblib")

# ---- A) unseen reviews ----
def predict_sentiment(text):
    return "Positive" if sentiment_model.predict([preprocess_review(text)])[0] == 1 else "Negative"

new_reviews = ["Amazing biryani and super friendly staff, loved the ambience!",
               "Food was cold, service was very slow and the waiter was rude. Never again.",
               "Decent place, nothing special"]
for r in new_reviews:
    print(f"{predict_sentiment(r):9s} <- {r}")

# ---- B) one unseen restaurant: which segment does it belong to? ----
def predict_cluster(raw):
    """raw = DataFrame with the base columns of a restaurant (same units as the training data)."""
    df = raw.copy()
    for c, (lo, hi) in bundle["cap_bounds"].items():
        df[c] = df[c].clip(lo, hi)                       # same outlier caps as in training
    df = add_new_features(df)
    X = df[bundle["features"]].copy()
    X[bundle["log_cols"]] = np.log1p(X[bundle["log_cols"]])
    return bundle["model"].predict(bundle["scaler"].transform(X))

new_restaurant = pd.DataFrame([{"Cost": 1500, "avg_rating": 4.3, "avg_sentiment": 0.6, "pct_negative": 4,
                                "avg_pictures": 1.2, "avg_review_words": 70, "avg_reviewer_followers": 300,
                                "n_cuisines": 4, "n_collections": 3}])
print("\nThe new restaurant belongs to cluster:", predict_cluster(new_restaurant)[0])

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

✏️ Edit this in your own words and fill in the numbers from your outputs.

- **Data:** two tables (105 restaurants, 10,000 reviews). After cleaning, 9,961 reviews and 100 restaurants were used.
- **EDA:** ratings are skewed to the top (about 62% are 4-5 stars) but there is a block of 1-star reviews (about 17%). The median cost is Rs. 700. Reviews peak in the late evening and at weekends. Premium restaurants rate higher on average.
- **Hypotheses:** cost and rating are positively related (rho about 0.38, significant), reviews with pictures rate higher (3.86 vs 3.54, significant), critics do not rate differently (not significant).
- **Clustering:** three models (K-Means, Agglomerative, GMM) were tuned and compared; K-Means with ✏️ [K] clusters was chosen (silhouette about ✏️ [value]). The clusters are *moderately* separated, so they should be read as overlapping market segments and not as hard boundaries. The segments are: ✏️ [name them].
- **Sentiment model:** the best classifier reached about ✏️ [macro-F1] macro-F1 and ✏️ [negative recall] recall on negative reviews on unseen test data.
- **Business impact:** customers can choose restaurants by segment; Zomato can promote hidden gems, support weak restaurants and personalise recommendations.
- **Limitations:** only 100 restaurants in one city; reviews are the latest 100 per restaurant, so no long-term trends; VADER is a general-purpose sentiment tool; cluster structure is moderate.
- **Future work:** add location data and menu prices, try DBSCAN and transformer-based sentiment models (e.g. BERT), and deploy the saved models behind a small web app.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***